<h1 style="color:#1f77b4; font-family:'Times New Roman';">
<b>SVD Intuition</b>
</h1>
<div style="font-family:'Times New Roman';">
SVD (Singular Value Decomposition) is one of those things that shows up everywhere, PCA, recommender systems, image compression, all of them use it under the hood. The idea is that any matrix A, doesnt matter what shape, can be broken into three pieces:
<br><br>
<b>A = U &Sigma; V<sup>T</sup></b>
<br><br>
i did a quick version of this back in the math foundations folder, but now i actually want to understand what these three pieces mean and why they are useful.
</div>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## What are the three pieces

If A is an m by n matrix, then:

- **U** (m by m) : the left singular vectors, basically directions in the output space
- **&Sigma;** : a diagonal matrix of singular values, sorted from biggest to smallest
- **V<sup>T</sup>** (n by n) : the right singular vectors, directions in the input space

The singular values on the diagonal of &Sigma; are the important bit. A big singular value means that direction carries a lot of the matrix, a tiny one means it barely matters. They always come out sorted, biggest first.

## A quick geometric way to think about it

Any matrix is just a linear transformation. SVD says that transformation is really just three simple steps: a rotation (V<sup>T</sup>), then a stretch along the axes (&Sigma;), then another rotation (U). So even a messy looking matrix is secretly just rotate, stretch, rotate.

In [2]:
# lets just do it on a small matrix and look at the pieces
A = np.array([[3, 1, 1],
              [-1, 3, 1]], dtype=float)

U, S, Vt = np.linalg.svd(A, full_matrices=False)
print('U shape:', U.shape)
print('singular values:', S.round(3))
print('Vt shape:', Vt.shape)

U shape: (2, 2)
singular values: [3.464 3.162]
Vt shape: (2, 3)


Notice the singular values are sorted big to small. Now the cool part, we can rebuild A by multiplying the pieces back together, and we can also rebuild only *part* of it by keeping just the top singular values.

In [4]:
# full reconstruction should give back A
A_full = U @ np.diag(S) @ Vt
print('reconstruction matches A?', np.allclose(A, A_full))

reconstruction matches A? True


<h2 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Low rank approximation (the actually useful part)</b>
</h2>
<div style="font-family:'Times New Roman';">
Here is the whole reason SVD matters for compression. Each singular value plus its U and V columns adds one more layer of detail to the matrix. The first few layers (the big singular values) carry most of the matrix, the rest are small tweaks. So if we keep only the top k and throw away the small ones, we get a very close approximation of A using way less numbers. That is a rank k approximation.
</div>

In [5]:
# rebuild A using only the top k singular values
def rank_k(U, S, Vt, k):
    return U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

for k in [1, 2]:
    approx = rank_k(U, S, Vt, k)
    err = np.linalg.norm(A - approx)
    print(f'k={k}  error={err:.3f}')

k=1  error=3.162
k=2  error=0.000


Even with k=1 (just the single biggest singular value) we already get pretty close, and k=2 nails it here because this little matrix only has 2 singular values anyway.

## How this connects to PCA

This is basically what PCA was doing. If you center your data and run SVD on it, the right singular vectors (V) are the principal components and the singular values are tied to how much variance each component explains. So PCA is kind of SVD wearing a different hat.

Next i'll build SVD from scratch so its not just a black box numpy call, and after that use it to compress an actual image.